In [58]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

In [59]:
with open('input.txt', 'r') as file:
    text = file.read()

# Get the unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(len(chars))


# Creating a mapping from characters to integers
stoi = { ch:i for i, ch in enumerate(chars) }
print(stoi)

# Creating a mapping from integers to characters
itos = { i:ch for i, ch in enumerate(chars) }
print(itos)

# Encode the text into integers
test = "hello"
encode = lambda s: [stoi[c] for c in s]
print(encode(test))

decode = lambda l: ''.join([itos[i] for i in l])
print(decode(encode(test)))




 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65
{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40

In [60]:
# Encode the whole corpus
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

print(train_data.shape)
print(val_data.shape)

torch.Size([1003854])
torch.Size([111540])


In [61]:
context_length = 8
batch_size = 32

train_data[:context_length+1]



tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [62]:
x = train_data[:context_length]
y = train_data[1:context_length+1]

for t in range(len(x)):
    context = x[t:t+context_length]
    target = y[t]
    print(f"when input is {context} the target is {target}")

when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 47
when input is tensor([47, 56, 57, 58,  1, 15, 47]) the target is 56
when input is tensor([56, 57, 58,  1, 15, 47]) the target is 57
when input is tensor([57, 58,  1, 15, 47]) the target is 58
when input is tensor([58,  1, 15, 47]) the target is 1
when input is tensor([ 1, 15, 47]) the target is 15
when input is tensor([15, 47]) the target is 47
when input is tensor([47]) the target is 58


In [63]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    """
    Randomly samples a batch of data from the dataset
    """
    data = train_data if split == 'train' else val_data
    random_indices = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in random_indices])
    y = torch.stack([data[i+1:i+block_size+1] for i in random_indices])
    return x, y

xb, yb = get_batch('train')
print(xb.shape)
print(xb)
print(yb.shape)
print(yb)
for b in range(batch_size):
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context} the target is {target}")




torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input is tensor([24]) the target is 43
when input is tensor([24, 43]) the target is 58
when input is tensor([24, 43, 58]) the target is 5
when input is tensor([24, 43, 58,  5]) the target is 57
when input is tensor([24, 43, 58,  5, 57]) the target is 1
when input is tensor([24, 43, 58,  5, 57,  1]) the target is 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]) the target is 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target is 39
when input is tensor([44]) the target is 53
when input is tensor([44, 53]) the target is 56
when input is tensor([44, 53, 56]) the target is 1
when input is tenso

In [64]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        """
        idx and targets are both (B, T) tensor of integers
        """
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None 
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1)
        return idx


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=30)[0].tolist()))


torch.Size([32, 65])
tensor(5.0364, grad_fn=<NllLossBackward0>)

lfJeukRuaRJKXAYtXzfJ:HEPiu--sD


In [67]:
# Math trick for self-attention

# Goal is to get x[b,t] = mean_{i<=t} x[b,i]
B,T,C = 4,8,2
x = torch.randn(B,T,C)

xbow = torch.randn((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = xprev.mean(dim=0)

# print(x[0])
# print(xbow[0])

# Instead you can use matrix multiplication
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, dim=1, keepdim=True) # Normalize the diagonal
b = torch.randint(0, 10, (3,2)).float()
c = a @ b
print("A")
print(a)
print("B")
print(b)
print("C")
print(c)



A
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
B
tensor([[9., 4.],
        [3., 3.],
        [0., 5.]])
C
tensor([[9.0000, 4.0000],
        [6.0000, 3.5000],
        [4.0000, 4.0000]])


In [70]:
# Version 2
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(dim=1, keepdim=True)
xbow2 = wei @ x
print(xbow2[0], xbow[0])

# Version 3
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T)) # Can think of this as a weight for previous tokens (they will be used in the self-attention)
wei = wei.masked_fill(tril == 0, float('-inf')) # This prevents the model from looking at future tokens in the decoder
wei = F.softmax(wei, dim=-1) # Softmax creates the averaging gradient in the matrix
xbow3 = wei @ x
print(xbow3[0], xbow[0])


tensor([[ 0.8022, -0.4510],
        [ 0.5462, -0.0864],
        [-0.0466,  0.0864],
        [-0.1148, -0.3739],
        [-0.0344, -0.4055],
        [-0.0969, -0.6108],
        [-0.0749, -0.4792],
        [-0.0890, -0.3675]]) tensor([[ 0.8022, -0.4510],
        [ 0.5462, -0.0864],
        [-0.0466,  0.0864],
        [-0.1148, -0.3739],
        [-0.0344, -0.4055],
        [-0.0969, -0.6108],
        [-0.0749, -0.4792],
        [-0.0890, -0.3675]])
tensor([[ 0.8022, -0.4510],
        [ 0.5462, -0.0864],
        [-0.0466,  0.0864],
        [-0.1148, -0.3739],
        [-0.0344, -0.4055],
        [-0.0969, -0.6108],
        [-0.0749, -0.4792],
        [-0.0890, -0.3675]]) tensor([[ 0.8022, -0.4510],
        [ 0.5462, -0.0864],
        [-0.0466,  0.0864],
        [-0.1148, -0.3739],
        [-0.0344, -0.4055],
        [-0.0969, -0.6108],
        [-0.0749, -0.4792],
        [-0.0890, -0.3675]])


In [80]:
class BatchNorm1d(nn.Module):
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        xmean = x.mean(dim=1, keepdim=True)
        xvar = x.var(dim=1, keepdim=True)
        xhat = (x - xmean) / (xvar + self.eps)**0.5
        return self.gamma * xhat + self.beta
    
    def parameters(self):
        return [self.gamma, self.beta]
    

module = BatchNorm1d(100)
x = torch.randn(32, 100)
print(module(x).shape)


torch.Size([32, 100])


In [76]:
# Version 3: Self-attention
# NOTE: This is a decoder block due to the tril function
# NOTE: For encoder blocks you can have connectivity between each node (remove the tril function)
# NOTE: This is self attention because the key, query, and value are all the same source (x)
# NOTE: For cross attention the key and value are from another source (like the encoder block) and the query is from the target
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, C) -> (B, T, head_size)
q = query(x) # (B, T, C) -> (B, T, head_size)

# NOTE: Normalizing the wei matrix is important for maintaining the gradient after the softmax function
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)

tril = torch.tril(torch.ones(T,T))
# wei = torch.zeros((T,T)) # Can think of this as a weight for previous tokens (they will be used in the self-attention)
wei = wei.masked_fill(tril == 0, float('-inf')) # This prevents the model from looking at future tokens in the decoder
wei = F.softmax(wei, dim=-1) # Softmax creates the averaging gradient in the matrix
print("WEI")
print(wei[0])
v = value(x)
out = wei @ v
print(out[0])


WEI
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3713, 0.6287, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6390, 0.3155, 0.0455, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6393, 0.0906, 0.0029, 0.2672, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2190, 0.1716, 0.1367, 0.2631, 0.2096, 0.0000, 0.0000, 0.0000],
        [0.4581, 0.0670, 0.0021, 0.1783, 0.1657, 0.1288, 0.0000, 0.0000],
        [0.0983, 0.1390, 0.2559, 0.1148, 0.1176, 0.1217, 0.1528, 0.0000],
        [0.1059, 0.1324, 0.1706, 0.0957, 0.1123, 0.0993, 0.1378, 0.1459]],
       grad_fn=<SelectBackward0>)
tensor([[-0.1074,  0.1780,  0.0395, -0.0236, -0.4260,  0.1737, -0.0575, -0.0628,
          0.1070, -0.2898, -0.1986, -0.6090, -0.1574,  0.4405,  0.3183,  0.6602],
        [ 0.0974,  0.2786, -0.0571, -0.0059, -0.3338, -0.0408, -0.0400,  0.0269,
          0.1313, -0.1096,  0.0088, -0.2664, -0.2798,  0.1672,  0.1747,  0.2000],
        [ 0.0008,  0.2006, -0.0101, -0.0124

In [48]:
# Train the Bigram Language Model
optimizer = torch.optim.AdamW(m.parameters(), lr=3e-2)
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if steps % 100 == 0:
        print(f"step {steps} loss {loss.item()}")

step 0 loss 2.368204355239868
step 100 loss 2.439711570739746
step 200 loss 2.358778476715088
step 300 loss 2.2869374752044678
step 400 loss 2.3858706951141357
step 500 loss 2.6149003505706787
step 600 loss 2.60050106048584
step 700 loss 2.5139248371124268
step 800 loss 2.4590532779693604
step 900 loss 2.421734094619751
step 1000 loss 2.4640145301818848
step 1100 loss 2.4631264209747314
step 1200 loss 2.3199870586395264
step 1300 loss 2.47470760345459
step 1400 loss 2.344618558883667
step 1500 loss 2.451127290725708
step 1600 loss 2.399040460586548
step 1700 loss 2.494954824447632
step 1800 loss 2.4619009494781494
step 1900 loss 2.3374860286712646
step 2000 loss 2.411619186401367
step 2100 loss 2.551198959350586
step 2200 loss 2.4294354915618896
step 2300 loss 2.32120680809021
step 2400 loss 2.4596335887908936
step 2500 loss 2.5857155323028564
step 2600 loss 2.532855987548828
step 2700 loss 2.489363193511963
step 2800 loss 2.406205654144287
step 2900 loss 2.4998438358306885
step 3000 l

In [49]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


CED thispon theh beleve ellarns:
NG pr aplve whoupaweagist Farie faringayourall REEves?
AG t yomyothive ot meryaf?

somy h'se, me VINGor, IUCat oues S:
t Myownd orupalveancestthefe ofrthes bowa y gh theear ord oum.

Mal INThar.
ARThhem,


In

I ngondr r buto g mme s fumid y ll ng, both
BRitellploure
